In [9]:
# === Загружаем данные ===
df = pd.read_excel(r"C:\Users\user\Documents\IMBIT\my_data.xlsx", sheet_name=0)

group1 = df["group1"].dropna().values
group2 = df["group2"].dropna().values

# Создадим таблицу для результатов
results = pd.DataFrame(columns=["Group", "n", "mean", "std", "median", "IQR", "min", "max", 
                                "Shapiro_stat", "Shapiro_p", "DAG_stat", "DAG_p", 
                                "MannWhitney_U", "MannWhitney_p", "Cliffs_delta", "Cliffs_interpret", 
                                "Cohen_d", "Cohen_interpret", "Hedges_g", "Hedges_interpret"])

# === Базовая описательная статистика ===
def describe(x):
    return np.mean(x), np.std(x, ddof=1), np.median(x), np.percentile(x, 75)-np.percentile(x, 25), np.min(x), np.max(x)

# === Нормальность ===
def normality_tests(x):
    shapiro_stat, shapiro_p = stats.shapiro(x)
    dag_stat, dag_p = (np.nan, np.nan)
    if len(x) >= 20:
        dag_stat, dag_p = stats.normaltest(x)
    return shapiro_stat, shapiro_p, dag_stat, dag_p

# === Cliff’s delta ===
def cliffs_delta(x, y):
    n, m = len(x), len(y)
    more = sum(1 for xi, yi in product(x, y) if xi > yi)
    less = sum(1 for xi, yi in product(x, y) if xi < yi)
    delta = (more - less) / (n * m)
    return delta

def interpret_cliffs(delta):
    d = abs(delta)
    if d < 0.147:
        return "незначительный эффект"
    elif d < 0.33:
        return "малый эффект"
    elif d < 0.474:
        return "средний эффект"
    else:
        return "большой эффект"

# === Cohen’s d и Hedges’ g ===
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx-1)*np.var(x, ddof=1) + (ny-1)*np.var(y, ddof=1)) / (nx+ny-2))
    return (np.mean(x) - np.mean(y)) / pooled_std

def hedges_g(d, nx, ny):
    correction = 1 - (3 / (4*(nx+ny) - 9))
    return d * correction

def interpret_effect(value):
    abs_val = abs(value)
    if abs_val < 0.2:
        return "незначительный эффект"
    elif abs_val < 0.5:
        return "малый эффект"
    elif abs_val < 0.8:
        return "средний эффект"
    else:
        return "большой эффект"

# === Mann–Whitney U-test ===
u_stat, u_p = stats.mannwhitneyu(group1, group2, alternative="two-sided")

# Заполнение таблицы для каждой группы
for name, group in zip(["Group1", "Group2"], [group1, group2]):
    mean, std, median, iqr, min_val, max_val = describe(group)
    sh_stat, sh_p, dag_stat, dag_p = normality_tests(group)
    d = cohens_d(group1, group2)
    g = hedges_g(d, len(group1), len(group2))
    delta = cliffs_delta(group1, group2)

    new_row = pd.DataFrame([{
        "Group": name,
        "n": len(group),
        "mean": mean,
        "std": std,
        "median": median,
        "IQR": iqr,
        "min": min_val,
        "max": max_val,
        "Shapiro_stat": sh_stat,
        "Shapiro_p": sh_p,
        "DAG_stat": dag_stat,
        "DAG_p": dag_p,
        "MannWhitney_U": u_stat,
        "MannWhitney_p": u_p,
        "Cliffs_delta": delta,
        "Cliffs_interpret": interpret_cliffs(delta),
        "Cohen_d": d,
        "Cohen_interpret": interpret_effect(d),
        "Hedges_g": g,
        "Hedges_interpret": interpret_effect(g)
    }])

    results = pd.concat([results, new_row], ignore_index=True)

# === Сохраняем результаты в Excel ===
results.to_excel(r"C:\Users\user\Documents\IMBIT\TRN Depolarisation\statistical_results_FWHM.xlsx", index=False)
print("Результаты сохранены в statistical_results_Cm.xlsx")


Результаты сохранены в statistical_results_Cm.xlsx


In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import product

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import product

In [ ]:
import pandas as pd

In [1]:
import numpy as np